In [ ]:
import glob
import logging
import logging.config
import os

import pandas as pd

from graal.summary.blind_eval_project import BlindEvalProject

logging.config.fileConfig("logging.conf")

DATA_FOLDER = os.getenv("DATA_FOLDER", "data")
RESPONSES_FOLDER = f"{DATA_FOLDER}/blind_eval_summaries/responses feb 11/"
PROJECT_SAVE_LOCATION = (
    f"{DATA_FOLDER}/blind_eval_summaries/project_eval_aveugle_objets.pkl"
)
EXCEL_OUTPUT_FILE = (
    f"{DATA_FOLDER}/blind_eval_summaries/resultat_eval_aveugle_feb_11.xlsx"
)
# PROJECT_SAVE_LOCATION = f"{DATA_FOLDER}/blind_eval_summaries/project_eval_aveugle_objets copy.pkl"
METRICS = ["Correct", "Complet", "Concis"]
OUTPUT_COLUMNS = ["ID", "Source"] + METRICS

result_df = pd.DataFrame(columns=OUTPUT_COLUMNS)

project = BlindEvalProject.load_from_disk(PROJECT_SAVE_LOCATION)
logging.info(f"Project '{PROJECT_SAVE_LOCATION}' successfully loaded")
display(project.mapping_obj_to_author)

# # Convert the dictionary to a list of dictionaries
# rows = [{"ID": k, **v} for k, v in project.mapping_obj_to_author.items()]

# # Create the DataFrame
# mapping_df = pd.DataFrame(rows)

# # Display the DataFrame
# print(mapping_df)
# mapping_df[mapping_df["ID"] <= 54].to_excel(
#     "correspondance_objet_auteur.xlsx", engine="openpyxl", index=False
# )

# Get all Excel files in the RESPONSES_FOLDER
excel_files = [
    file
    for file in glob.glob(os.path.join(RESPONSES_FOLDER, "*.xlsx"))
    if not os.path.basename(file).startswith("~")
]

# Read each Excel file into a dataframe and store them in a list
dataframes = [pd.read_excel(file, engine="openpyxl") for file in excel_files]
# dataframes = [pd.read_excel(file, engine="openpyxl") for file in excel_files[:1]]

# Display the list of dataframes
preferred_source_count = {}

for i, df in enumerate(dataframes):
    print(f"DataFrame {i} from file {excel_files[i]}:")
    for metric in METRICS:
        for col in [f"1 - {metric}", f"2 - {metric}"]:
            df[col] = df[col].fillna("non").apply(lambda x: x.lower())
            df[col] = df[col].apply(lambda x: 1 if x == "oui" else 0).astype(int)
    df["Objet 1"] = df["ID"].apply(
        lambda x: project.mapping_obj_to_author.get(x, {}).get("Objet 1", "")
    )
    df["Objet 2"] = df["ID"].apply(
        lambda x: project.mapping_obj_to_author.get(x, {}).get("Objet 2", "")
    )
    rows_to_append = []
    for _index, row in df.iterrows():
        row_to_append_obj1 = {"ID": row["ID"], "Source": row["Objet 1"]}
        row_to_append_obj2 = {"ID": row["ID"], "Source": row["Objet 2"]}
        for metric in METRICS:
            row_to_append_obj1[metric] = row[f"1 - {metric}"]
            row_to_append_obj2[metric] = row[f"2 - {metric}"]
        rows_to_append.append(row_to_append_obj1)
        rows_to_append.append(row_to_append_obj2)

        # Track the source choice count
        chosen_obj = row[
            "Au regard des trois critères présentés et selon vous, quel est le meilleur objet ? (1 ou 2)"
        ]
        if chosen_obj in row:
            chosen_source = row[chosen_obj]
        else:
            continue

        preferred_source_count[chosen_source] = (
            preferred_source_count.get(chosen_source, 0) + 1
        )

    result_df = pd.concat([result_df, pd.DataFrame(rows_to_append)], ignore_index=True)
    # display(df)


# display(result_df)
# Sum up the metrics columns of result_df grouped by "ID" and "Source"
summary_df = result_df.groupby(["ID", "Source"]).sum().reset_index()

# Display the summary dataframe
# display(summary_df)

# Save the summary dataframe to an Excel file
with pd.ExcelWriter(EXCEL_OUTPUT_FILE, engine="openpyxl", mode="w") as writer:
    summary_df.to_excel(writer, sheet_name="Metrics", index=False)
    pd.DataFrame.from_dict(
        preferred_source_count, orient="index", columns=["Count"]
    ).to_excel(writer, sheet_name="Expert vs LLMs")

logging.info(f"Summary saved to '{EXCEL_OUTPUT_FILE}'")

os.system(f'open "{EXCEL_OUTPUT_FILE}"')

INFO - Project 'data/blind_eval_summaries/project_eval_aveugle_objets.pkl' successfully loaded


{0: {'Objet 1': 'Expert', 'Objet 2': 'llama-3.1-70b-instruct'},
 1: {'Objet 1': 'gpt-4-turbo', 'Objet 2': 'Expert'},
 2: {'Objet 1': 'Expert', 'Objet 2': 'gpt-4o-mini'},
 3: {'Objet 1': 'llama-3.1-70b-instruct', 'Objet 2': 'Expert'},
 4: {'Objet 1': 'Expert', 'Objet 2': 'gpt-4-turbo'},
 5: {'Objet 1': 'Expert', 'Objet 2': 'gpt-4o-mini'},
 6: {'Objet 1': 'llama-3.1-70b-instruct', 'Objet 2': 'Expert'},
 7: {'Objet 1': 'Expert', 'Objet 2': 'gpt-4-turbo'},
 8: {'Objet 1': 'gpt-4o-mini', 'Objet 2': 'Expert'},
 9: {'Objet 1': 'Expert', 'Objet 2': 'llama-3.1-70b-instruct'},
 10: {'Objet 1': 'Expert', 'Objet 2': 'gpt-4-turbo'},
 11: {'Objet 1': 'Expert', 'Objet 2': 'gpt-4o-mini'},
 12: {'Objet 1': 'Expert', 'Objet 2': 'llama-3.1-70b-instruct'},
 13: {'Objet 1': 'gpt-4-turbo', 'Objet 2': 'Expert'},
 14: {'Objet 1': 'Expert', 'Objet 2': 'gpt-4o-mini'},
 15: {'Objet 1': 'llama-3.1-70b-instruct', 'Objet 2': 'Expert'},
 16: {'Objet 1': 'gpt-4-turbo', 'Objet 2': 'Expert'},
 17: {'Objet 1': 'Expert',

    ID                 Objet 1                 Objet 2
0    0                  Expert  llama-3.1-70b-instruct
1    1             gpt-4-turbo                  Expert
2    2                  Expert             gpt-4o-mini
3    3  llama-3.1-70b-instruct                  Expert
4    4                  Expert             gpt-4-turbo
..  ..                     ...                     ...
75  75                  Expert  llama-3.1-70b-instruct
76  76                  Expert             gpt-4-turbo
77  77             gpt-4o-mini                  Expert
78  78  llama-3.1-70b-instruct                  Expert
79  79                  Expert             gpt-4-turbo

[80 rows x 3 columns]
DataFrame 0 from file data/blind_eval_summaries/responses feb 11/20241220_eval_aveugle_objets_TR.xlsx:
DataFrame 1 from file data/blind_eval_summaries/responses feb 11/20241220_eval_aveugle_objets_VEB.xlsx:
DataFrame 2 from file data/blind_eval_summaries/responses feb 11/20241220_eval_aveugle_objets_VDEF_C_BORIAUD.x

0